In [4]:
import csv
import json
import random
from datetime import date, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260624
NUMBER_OF_CONTRACTS = 20_000

random.seed(RANDOM_SEED)


# ============================================================
# PROJECT PATHS
# ============================================================

# Jupyter is already running inside risk_scoring_project,
# so use its current folder directly.
PROJECT_ROOT = Path.cwd()

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# REFERENCE DATA
# ============================================================

BRANCHES = [
    "001", "007", "015", "022", "031",
    "042", "056", "071", "089", "102",
    "118", "145", "168", "201", "268"
]

PRODUCTS = [
    {"product_code": "CZL3", "product_category": "CONSUMER_LOAN"},
    {"product_code": "AUTO", "product_category": "AUTO_LOAN"},
    {"product_code": "MORT", "product_category": "MORTGAGE"},
    {"product_code": "BIZL", "product_category": "SME_LOAN"},
    {"product_code": "PERS", "product_category": "PERSONAL_LOAN"},
    {"product_code": "CRED", "product_category": "CREDIT_LINE"},
]

CUSTOMER_IDS = [str(customer_id) for customer_id in range(4_000_001, 4_015_001)]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def random_date(start_date: date, end_date: date) -> date:
    """Returns one random date between two dates."""

    number_of_days = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, number_of_days))


def generate_account_number(
    branch_code: str,
    product_code: str,
    book_date: date,
    sequence_number: int
) -> str:
    """
    Creates a 16-character account number.

    Example:
    268CZL3171380001
    """

    year_part = book_date.strftime("%y")
    sequence_part = f"{sequence_number:07d}"

    return f"{branch_code}{product_code}{year_part}{sequence_part}"


def generate_application_number(
    book_date: date,
    sequence_number: int
) -> str:
    """
    Generates a maximum 16-character application number.
    Example: APP2600000000001
    """

    return f"APP{book_date:%y}{sequence_number:011d}"


def generate_contract_row(sequence_number: int) -> dict:
    """
    Generates one clean contract-level record.

    These fields are the identity and relationship fields needed
    before we generate the remaining loan attributes.
    """

    branch_code = random.choice(BRANCHES)
    product = random.choice(PRODUCTS)

    book_date = random_date(
        start_date=date(2018, 1, 1),
        end_date=date(2026, 5, 31)
    )

    account_number = generate_account_number(
        branch_code=branch_code,
        product_code=product["product_code"],
        book_date=book_date,
        sequence_number=sequence_number
    )

    customer_id = random.choice(CUSTOMER_IDS)

    return {
        "ACCOUNT_NUMBER": account_number,
        "BRANCH_CODE": branch_code,
        "APPLICATION_NUM": generate_application_number(
            book_date=book_date,
            sequence_number=sequence_number
        ),
        "CUSTOMER_ID": customer_id,
        "PRODUCT_CODE": product["product_code"],
        "PRODUCT_CATEGORY": product["product_category"],
        "BOOK_DATE": book_date.isoformat(),
        "VALUE_DATE": book_date.isoformat(),
        "ORIGINAL_ST_DATE": book_date.isoformat(),
        "PRIMARY_APPLICANT_ID": customer_id,
        "ALT_ACC_NO": account_number
    }


def validate_contracts(contract_rows: list[dict]) -> None:
    """
    Stops the script if contract identifiers are invalid.
    """

    account_numbers = [
        row["ACCOUNT_NUMBER"]
        for row in contract_rows
    ]

    application_numbers = [
        row["APPLICATION_NUM"]
        for row in contract_rows
    ]

    if len(contract_rows) != NUMBER_OF_CONTRACTS:
        raise ValueError(
            f"Expected {NUMBER_OF_CONTRACTS:,} rows, "
            f"but generated {len(contract_rows):,}."
        )

    if len(account_numbers) != len(set(account_numbers)):
        raise ValueError("Duplicate ACCOUNT_NUMBER values were generated.")

    if len(application_numbers) != len(set(application_numbers)):
        raise ValueError("Duplicate APPLICATION_NUM values were generated.")

    if any(not account_number for account_number in account_numbers):
        raise ValueError("A blank ACCOUNT_NUMBER was generated.") 

    if any(len(account_number) > 35 for account_number in account_numbers):
        raise ValueError(
            "At least one ACCOUNT_NUMBER exceeds VARCHAR2(35)."
        )


def write_csv(file_path: Path, contract_rows: list[dict]) -> None:
    """Writes contract data to CSV."""

    column_names = list(contract_rows[0].keys())

    with open(file_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=column_names
        )

        writer.writeheader()
        writer.writerows(contract_rows)


# ============================================================
# MAIN SCRIPT
# ============================================================

def main() -> None:
    print(f"Generating {NUMBER_OF_CONTRACTS:,} unique contracts...")

    contract_rows = [
        generate_contract_row(sequence_number=sequence_number)
        for sequence_number in range(1, NUMBER_OF_CONTRACTS + 1)
    ]

    validate_contracts(contract_rows)

    full_output_path = RAW_DATA_FOLDER / "rs_contracts_base.csv"
    sample_output_path = SAMPLE_DATA_FOLDER / "rs_contracts_base_sample.csv"
    summary_output_path = RAW_DATA_FOLDER / "rs_contracts_generation_summary.json"

    write_csv(
        file_path=full_output_path,
        contract_rows=contract_rows
    )

    write_csv(
        file_path=sample_output_path,
        contract_rows=contract_rows[:100]
    )

    generation_summary = {
        "table_name": "RS_CONTRACTS",
        "generated_rows": len(contract_rows),
        "unique_account_numbers": len(
            {row["ACCOUNT_NUMBER"] for row in contract_rows}
        ),
        "unique_application_numbers": len(
            {row["APPLICATION_NUM"] for row in contract_rows}
        ),
        "customer_id_pool_size": len(CUSTOMER_IDS),
        "random_seed": RANDOM_SEED,
        "full_file": str(full_output_path),
        "sample_file": str(sample_output_path)
    }

    with open(summary_output_path, "w", encoding="utf-8") as file:
        json.dump(
            generation_summary,
            file,
            indent=4
        )

    print("Generation completed successfully.")
    print(f"Full dataset: {full_output_path}")
    print(f"Sample dataset: {sample_output_path}")
    print(f"Unique ACCOUNT_NUMBER count: {generation_summary['unique_account_numbers']:,}")


if __name__ == "__main__":
    main()

Generating 20,000 unique contracts...
Generation completed successfully.
Full dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_contracts_base.csv
Sample dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_contracts_base_sample.csv
Unique ACCOUNT_NUMBER count: 20,000


In [2]:
from pathlib import Path

print(Path.cwd())

c:\Users\ASUS\Desktop\project\risk_scoring_project


In [5]:
import csv
import json
import random
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260625
NUMBER_OF_CUSTOMERS = 15_000

# This matches the customer ID pool used in generate_contracts.py.
CUSTOMER_ID_START = 4_000_001
CUSTOMER_ID_END = 4_015_000

random.seed(RANDOM_SEED)


# ============================================================
# NOTEBOOK-SAFE PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

CONTRACT_FILE = PROJECT_ROOT / "data" / "raw" / "rs_contracts_base.csv"
RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# CUSTOMER TABLE COLUMNS
# ============================================================

CUSTOMER_COLUMNS = [
    "GEN_DIM_CUST_ID",
    "CUSTOMER_NO",
    "PIN",
    "DOCUMENT_NO",
    "TAX_ID",
    "CUSTOMER_TYPE",
    "CUSTOMER_SUB_TYPE",
    "FULL_NAME",
    "FIRST_NAME",
    "LAST_NAME",
    "MIDDLE_NAME",
    "GENDER",
    "BIRTH_DT",
    "PLACE_OF_BIRTH",
    "MARITAL_STATUS",
    "NATIONALITY",
    "OPEN_REASON",
    "CREATION_DT",
    "OPENING_BRN",
    "FLG_IS_COURT_ISSUE",
    "OCCUPATION",
    "WORKPLACE_CUSTOMER_NO",
    "WORKPLACE_NAME",
    "FLG_IS_BLACK_LIST",
    "FLG_IS_VIP",
    "FLG_IS_EMPLOYEER",
    "NUM_OF_EMPLOYEER",
    "RM_NAME",
    "CUSTOMER_GENERATION",
    "FLG_IS_AFFLUENT",
    "FLG_IS_SALARY",
    "FLG_IS_PENSION",
    "FLG_IS_EDV",
    "RISK_LEVEL",
    "RESIDENT_STATUS",
    "RM_ID_CORP",
    "GROUP_ID_CORP",
    "CHECKER_DT_STAMP",
    "MAKER_DT_STAMP",
    "EMPL_PROPERTY_TYPE",
    "SHORT_NAME",
    "COUNTRY",
    "DOCUMENT_NAME",
    "LEGAL_GUARDIAN",
    "PASSPORT_COUNTRY",
    "PASSPORT_NATIONAL_ID",
    "PASSPORT_ISS_DT",
    "PASSPORT_EXP_DT",
    "P_ADDRESS1",
    "P_ADDRESS2",
    "P_ADDRESS3",
    "ENGLISH_FULL_NAME",
    "FLG_IS_BENEFICIARY",
    "CUSTOMER_SINCE",
    "DIRECTOR_CUST_NO",
    "FLG_IS_GREEN_CARD",
    "US_BIRTH",
    "FLG_IS_US_CITIZEN",
    "US_PAYMENT",
    "FLG_IS_US_RESIDENT_POA",
    "FLG_IS_US_ADDRESS",
    "FLG_IS_US_LIVING",
    "US_PASSPORT",
    "FLG_IS_US_PHONE",
    "FLG_IS_US_POA",
    "FLG_IS_US_POST_BOX",
    "US_RELATED_PERSON",
    "FLG_IS_US_TAX_RESIDENT",
    "WORKPLACE_ADDRESS",
    "WORKPLACE",
    "CIF_RECORD_STATUS",
    "FLG_IS_FROZEN",
    "INS_DTTM",
    "TAX_TYPE",
    "FLG_IS_INGROUP",
    "CUST_MIS_1",
    "SHORT_NAME2",
    "SWIFT_CD",
    "CHARGE_GROUP",
    "DEFAULT_MEDIA",
    "CUSTOMER_ADDRESS",
    "CORP_REGISTER_ADDRESS",
    "CORP_REGISTER_COUNTRY",
    "CHECKER_DT",
    "FLG_IS_MILITARY_PASSPORT",
    "CUST_SUB_SEGMENT",
    "AGRAR_PKD",
    "FLG_IS_ACTIVE_CUSTOMER_QUARTERLY",
    "FLG_IS_REMOTE_ACC_ALLOWED",
    "ONLY_LAST_AND_FIRST_NAME_ENG",
    "EMPLOYER_ID",
    "INCORP_DATE",
    "DT_QHT_GROUP",
    "CUST_CLASSIFICATION",
    "LOAN_RESP_ID",
    "ULTIMATE_BENEFICIAL_OWNER",
    "FLG_IS_RELATED_PERSON",
    "COUNTRY_DESC",
    "REGISTRATION_ADDRESS",
    "DOMICILE_ADDRESS",
]


# ============================================================
# REFERENCE DATA
# ============================================================

BRANCHES = [
    "001", "007", "015", "022", "031",
    "042", "056", "071", "089", "102",
    "118", "120", "145", "168", "201", "268"
]

CITIES = [
    "BAKI", "SUMQAYIT", "GANCA", "MINGECEVIR", "SIRVAN",
    "LENKERAN", "QUBA", "SEKI", "MASALLI", "SAMAXI",
    "YEVLAX", "QABALA", "NAXCIVAN", "ZAGATALA"
]

STREETS = [
    "NIZAMI KUCHESI",
    "ATATURK PROSPEKTI",
    "HEYDER ELIYEV PROSPEKTI",
    "XETAI PROSPEKTI",
    "TBLISI PROSPEKTI",
    "BABEK PROSPEKTI",
    "SULEYMAN RUSTEM KUCHESI",
    "AZADLIQ PROSPEKTI"
]

MALE_FIRST_NAMES = [
    "Elvin", "Rashad", "Murad", "Tural", "Orkhan", "Kamran",
    "Nihad", "Samir", "Fuad", "Javid", "Anar", "Emil",
    "Rovshan", "Elnur", "Aydin", "Teymur", "Ramil", "Aqil"
]

FEMALE_FIRST_NAMES = [
    "Aysel", "Nigar", "Leyla", "Gunel", "Sevda", "Narmin",
    "Aynur", "Khadija", "Sabina", "Aytac", "Zahra", "Arzu",
    "Laman", "Ulker", "Fatima", "Rena", "Aysu", "Turan"
]

FATHER_NAMES = [
    "Rasim", "Vugar", "Camil", "Tahir", "Nazim", "Farhad",
    "Ilham", "Elman", "Adil", "Rauf", "Natiq", "Samad"
]

SURNAMES = [
    "Mammadov", "Aliyev", "Hasanov", "Huseynov", "Ibrahimov",
    "Rahimov", "Karimov", "Quliyev", "Babayeva", "Hajiyeva",
    "Asgarov", "Jafarov", "Suleymanov", "Nabiyev", "Ismayilov",
    "Abbasov", "Rzayev", "Safarov", "Tagiyev", "Aghayev"
]

OCCUPATIONS = [
    "ACCOUNTANT", "TEACHER", "ENGINEER", "DOCTOR",
    "SALES SPECIALIST", "IT SPECIALIST", "DRIVER",
    "MANAGER", "LAWYER", "ANALYST", "ENTREPRENEUR",
    "TECHNICIAN", "CIVIL SERVANT", "DESIGNER"
]

WORKPLACES = [
    "CASPIAN LOGISTICS MMC",
    "BAKU RETAIL MMC",
    "AZER TELECOM MMC",
    "NORTH CONSTRUCTION MMC",
    "CITY EDUCATION CENTER",
    "MEDICAL SERVICE MMC",
    "FINANCE CONSULTING MMC",
    "REGIONAL TRADE MMC",
    "TECH SOLUTIONS MMC"
]

COMPANY_PREFIXES = [
    "Caspian", "Araz", "Baku", "Shirvan", "Absheron",
    "Zirve", "Khazar", "North", "Silk Road", "Azer",
    "Green", "Atlas"
]

COMPANY_ACTIVITIES = [
    "Logistics", "Construction", "Trade", "Food",
    "Technology", "Textile", "Agriculture", "Pharmacy",
    "Tourism", "Energy", "Consulting", "Manufacturing",
    "Transport", "Services"
]

COMPANY_SUFFIXES = ["MMC", "ASC"]

RELATIONSHIP_MANAGERS = [
    "A. Mammadov",
    "L. Aliyeva",
    "R. Hasanov",
    "N. Ibrahimova",
    "K. Huseynov",
    "S. Rahimova"
]

SWIFT_CODES = [
    "AZEBAZ22",
    "BAKUAJ22",
    "CASPAZ22",
    "FINAAZ22"
]

PIN_ALPHABET = "ABCDEFGHJKLMNPQRSTUVWXYZ0123456789"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def random_date(start_date: date, end_date: date) -> date:
    if end_date < start_date:
        return start_date

    days_between = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, days_between))


def format_date(value):
    return value.isoformat() if value else ""


def format_timestamp(value):
    return value.strftime("%Y-%m-%d %H:%M:%S") if value else ""


def years_before(reference_date: date, years: int) -> date:
    try:
        return reference_date.replace(year=reference_date.year - years)
    except ValueError:
        return reference_date.replace(
            year=reference_date.year - years,
            month=2,
            day=28
        )


def generation_label(birth_date: date) -> str:
    if birth_date.year <= 1964:
        return "BABY_BOOMERS"
    if birth_date.year <= 1980:
        return "GEN_X"
    if birth_date.year <= 1996:
        return "MILLENNIALS_GEN_Y"

    return "GEN_Z"


def parse_iso_date(value: str):
    if not value:
        return None

    try:
        return datetime.strptime(value.strip(), "%Y-%m-%d").date()
    except ValueError:
        return None


def generate_unique_pin(existing_pins: set) -> str:
    while True:
        pin = (
            str(random.randint(1, 9))
            + "".join(random.choices(PIN_ALPHABET, k=6))
        )

        if pin not in existing_pins:
            existing_pins.add(pin)
            return pin


def generate_unique_document_number(existing_documents: set) -> str:
    while True:
        document_number = f"AZE{random.randint(10_000_000, 99_999_999)}"

        if document_number not in existing_documents:
            existing_documents.add(document_number)
            return document_number


def generate_unique_tax_id(existing_tax_ids: set) -> str:
    while True:
        tax_id = str(random.randint(1_000_000_000, 9_999_999_999))

        if tax_id not in existing_tax_ids:
            existing_tax_ids.add(tax_id)
            return tax_id


def make_address(city: str) -> str:
    return (
        f"{city}, {random.choice(STREETS)}, "
        f"EV {random.randint(1, 250)}, MEN {random.randint(1, 150)}"
    )


# ============================================================
# CONTRACT READING AND KEY MATCHING
# ============================================================

def load_contract_customer_details():
    """
    Reads contracts and collects:
    - customer IDs
    - earliest contract date per customer
    - products per customer
    """

    if not CONTRACT_FILE.exists():
        raise FileNotFoundError(
            f"Contract file not found: {CONTRACT_FILE}\n"
            "First run the contract generator."
        )

    customer_ids = set()
    first_contract_date = {}
    product_codes = defaultdict(set)

    with open(
        CONTRACT_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "CUSTOMER_ID",
            "BOOK_DATE",
            "PRODUCT_CODE"
        }

        missing_columns = required_columns - set(reader.fieldnames or [])

        if missing_columns:
            raise ValueError(
                "Missing contract columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            customer_id = (row.get("CUSTOMER_ID") or "").strip()

            if not customer_id:
                continue

            customer_ids.add(customer_id)

            book_date = parse_iso_date(
                row.get("BOOK_DATE") or ""
            )

            if book_date:
                current_minimum = first_contract_date.get(customer_id)

                if (
                    current_minimum is None
                    or book_date < current_minimum
                ):
                    first_contract_date[customer_id] = book_date

            product_code = (row.get("PRODUCT_CODE") or "").strip()

            if product_code:
                product_codes[customer_id].add(product_code)

    return customer_ids, first_contract_date, product_codes


def build_customer_numbers(contract_customer_ids: set) -> list:
    """
    Creates exactly 15,000 CIF values.

    Every CUSTOMER_ID in contracts is included here, guaranteeing:
    CUSTOMER_ID = CUSTOMER_NO.
    """

    customer_numbers = [
        str(customer_id)
        for customer_id in range(
            CUSTOMER_ID_START,
            CUSTOMER_ID_END + 1
        )
    ]

    if len(customer_numbers) != NUMBER_OF_CUSTOMERS:
        raise ValueError(
            "Customer ID range must create exactly "
            f"{NUMBER_OF_CUSTOMERS:,} records."
        )

    unmatched_contract_ids = (
        contract_customer_ids - set(customer_numbers)
    )

    if unmatched_contract_ids:
        examples = ", ".join(sorted(unmatched_contract_ids)[:5])

        raise ValueError(
            "Some contract CUSTOMER_ID values are outside the expected "
            f"customer-ID range. Examples: {examples}"
        )

    random.shuffle(customer_numbers)

    return customer_numbers


def choose_customer_type(product_codes: set) -> str:
    """
    Business loans produce more corporate customers.
    Consumer loans produce mostly individual customers.
    """

    if "BIZL" in product_codes:
        return random.choices(
            ["I", "C", "B"],
            weights=[35, 63, 2],
            k=1
        )[0]

    if product_codes:
        return random.choices(
            ["I", "C", "B"],
            weights=[88, 11, 1],
            k=1
        )[0]

    return random.choices(
        ["I", "C", "B"],
        weights=[80, 19, 1],
        k=1
    )[0]


# ============================================================
# ROW GENERATORS
# ============================================================

def make_base_row(customer_no: str, generated_id: int) -> dict:
    row = {
        column_name: ""
        for column_name in CUSTOMER_COLUMNS
    }

    row.update({
        "GEN_DIM_CUST_ID": str(generated_id),
        "CUSTOMER_NO": customer_no,
        "FLG_IS_COURT_ISSUE": "N",
        "FLG_IS_BLACK_LIST": "N",
        "FLG_IS_VIP": "N",
        "FLG_IS_EMPLOYEER": "N",
        "FLG_IS_EDV": "Y",
        "FLG_IS_BENEFICIARY": "N",
        "FLG_IS_GREEN_CARD": "N",
        "US_BIRTH": "N",
        "FLG_IS_US_CITIZEN": "N",
        "US_PAYMENT": "N",
        "FLG_IS_US_RESIDENT_POA": "N",
        "FLG_IS_US_ADDRESS": "N",
        "FLG_IS_US_LIVING": "N",
        "US_PASSPORT": "N",
        "FLG_IS_US_PHONE": "N",
        "FLG_IS_US_POA": "N",
        "FLG_IS_US_POST_BOX": "N",
        "US_RELATED_PERSON": "N",
        "FLG_IS_US_TAX_RESIDENT": "N",
        "CIF_RECORD_STATUS": "O",
        "FLG_IS_FROZEN": "N",
        "FLG_IS_INGROUP": "N",
        "FLG_IS_MILITARY_PASSPORT": "N",
        "FLG_IS_ACTIVE_CUSTOMER_QUARTERLY": "Y",
        "FLG_IS_REMOTE_ACC_ALLOWED": "Y",
        "FLG_IS_RELATED_PERSON": "N",
        "COUNTRY": "AZ",
        "COUNTRY_DESC": "AZERBAIJAN",
        "PASSPORT_COUNTRY": "AZ",
        "RESIDENT_STATUS": "R",
        "RISK_LEVEL": str(
            random.choices(
                [1, 2, 3, 4, 5],
                weights=[10, 30, 35, 20, 5],
                k=1
            )[0]
        ),
        "DEFAULT_MEDIA": random.choice(
            ["SMS", "EMAIL", "MOBILE_APP"]
        ),
        "OPENING_BRN": random.choice(BRANCHES),
        "INS_DTTM": "2026-06-21",
        "CHECKER_DT": "2026-06-21",
    })

    return row


def generate_individual(
    row: dict,
    first_contract_date: date,
    existing_pins: set,
    existing_documents: set
) -> dict:
    gender = random.choice(["M", "F"])

    first_name = (
        random.choice(MALE_FIRST_NAMES)
        if gender == "M"
        else random.choice(FEMALE_FIRST_NAMES)
    )

    last_name = random.choice(SURNAMES)
    father_name = random.choice(FATHER_NAMES)

    middle_name = (
        f"{father_name} OGLU"
        if gender == "M"
        else f"{father_name} QIZI"
    )

    age_at_loan = random.randint(21, 72)

    birth_date = years_before(
        first_contract_date,
        age_at_loan
    )

    birth_date -= timedelta(days=random.randint(0, 364))

    customer_creation_start = max(
        date(2005, 1, 1),
        birth_date + timedelta(days=18 * 365)
    )

    creation_date = random_date(
        customer_creation_start,
        first_contract_date
    )

    passport_issue_date = random_date(
        max(
            birth_date + timedelta(days=16 * 365),
            date(2010, 1, 1)
        ),
        creation_date
    )

    passport_expiry_date = (
        passport_issue_date
        + timedelta(days=random.randint(7 * 365, 10 * 365))
    )

    city = random.choice(CITIES)
    occupation = random.choice(OCCUPATIONS)
    workplace = random.choice(WORKPLACES)

    is_salary_customer = random.choices(
        ["Y", "N"],
        weights=[58, 42],
        k=1
    )[0]

    is_pensioner = (
        "Y"
        if age_at_loan >= 63 and random.random() < 0.55
        else "N"
    )

    is_affluent = "Y" if random.random() < 0.08 else "N"

    is_vip = (
        "Y"
        if is_affluent == "Y" and random.random() < 0.12
        else "N"
    )

    is_employee = "Y" if random.random() < 0.02 else "N"

    full_name = f"{last_name} {first_name} {middle_name}"
    english_name = f"{last_name.upper()} {first_name.upper()}"

    row.update({
        "PIN": generate_unique_pin(existing_pins),
        "DOCUMENT_NO": generate_unique_document_number(
            existing_documents
        ),
        "CUSTOMER_TYPE": "I",
        "CUSTOMER_SUB_TYPE": random.choices(
            ["INDIVIDUAL", "RETAIL", "INDIVIDUAL OWNER"],
            weights=[50, 42, 8],
            k=1
        )[0],
        "FULL_NAME": full_name,
        "FIRST_NAME": first_name,
        "LAST_NAME": last_name,
        "MIDDLE_NAME": middle_name,
        "GENDER": gender,
        "BIRTH_DT": format_date(birth_date),
        "PLACE_OF_BIRTH": city,
        "MARITAL_STATUS": random.choices(
            ["S", "M", "D", "W"],
            weights=[35, 54, 7, 4],
            k=1
        )[0],
        "NATIONALITY": "AZ",
        "OPEN_REASON": (
            "PENSION"
            if is_pensioner == "Y"
            else "SALARY"
            if is_salary_customer == "Y"
            else random.choice(["CARD", "ABBM", "LOAN"])
        ),
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": occupation,
        "WORKPLACE_CUSTOMER_NO": (
            str(random.randint(5_000_001, 5_050_000))
            if random.random() < 0.45
            else ""
        ),
        "WORKPLACE_NAME": workplace,
        "FLG_IS_VIP": is_vip,
        "FLG_IS_EMPLOYEER": is_employee,
        "RM_NAME": (
            random.choice(RELATIONSHIP_MANAGERS)
            if is_affluent == "Y"
            else ""
        ),
        "CUSTOMER_GENERATION": generation_label(birth_date),
        "FLG_IS_AFFLUENT": is_affluent,
        "FLG_IS_SALARY": is_salary_customer,
        "FLG_IS_PENSION": is_pensioner,
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(
                hours=12,
                minutes=random.randint(0, 59)
            )
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(
                hours=11,
                minutes=random.randint(0, 59)
            )
        ),
        "EMPL_PROPERTY_TYPE": random.choice([
            "PRIVATE_SECTOR",
            "PUBLIC_SECTOR",
            "SELF_EMPLOYED"
        ]),
        "SHORT_NAME": (
            f"{first_name[:7].upper()}{row['PIN'][:7]}"
        )[:20],
        "DOCUMENT_NAME": "IDENTITY_CARD",
        "PASSPORT_NATIONAL_ID": row["PIN"],
        "PASSPORT_ISS_DT": format_date(passport_issue_date),
        "PASSPORT_EXP_DT": format_date(passport_expiry_date),
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": english_name,
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(
            random.choice(CITIES)
        ),
        "WORKPLACE": workplace,
        "CUST_MIS_1": "RETAIL",
        "SHORT_NAME2": (
            f"{last_name[:8].upper()}{first_name[:4].upper()}"
        )[:20],
        "CHARGE_GROUP": "FIZIKI",
        "CUSTOMER_ADDRESS": make_address(city),
        "CUST_SUB_SEGMENT": (
            "AFFLUENT"
            if is_affluent == "Y"
            else "MASS"
        ),
        "ONLY_LAST_AND_FIRST_NAME_ENG": english_name,
        "EMPLOYER_ID": (
            f"EMP{random.randint(1000, 9999)}"
            if is_employee == "Y"
            else ""
        ),
        "CUST_CLASSIFICATION": (
            "VIP"
            if is_vip == "Y"
            else "PREMIUM"
            if is_affluent == "Y"
            else "STANDARD"
        ),
    })

    return row


def generate_company(
    row: dict,
    first_contract_date: date,
    existing_tax_ids: set,
    company_sequence: int
) -> dict:
    company_name = (
        f"{random.choice(COMPANY_PREFIXES)} "
        f"{random.choice(COMPANY_ACTIVITIES)} "
        f"{random.choice(COMPANY_SUFFIXES)} "
        f"{company_sequence:04d}"
    )

    incorporation_date = random_date(
        date(1995, 1, 1),
        first_contract_date - timedelta(days=30)
    )

    creation_date = random_date(
        incorporation_date,
        first_contract_date
    )

    city = random.choice(CITIES)

    employee_count = random.choices(
        [
            random.randint(5, 49),
            random.randint(50, 249),
            random.randint(250, 3000)
        ],
        weights=[60, 32, 8],
        k=1
    )[0]

    is_affluent = (
        "Y"
        if employee_count >= 250 or random.random() < 0.18
        else "N"
    )

    row.update({
        "TAX_ID": generate_unique_tax_id(existing_tax_ids),
        "CUSTOMER_TYPE": "C",
        "CUSTOMER_SUB_TYPE": random.choices(
            ["SME/C", "CORPORATE"],
            weights=[72, 28],
            k=1
        )[0],
        "FULL_NAME": company_name,
        "OPEN_REASON": "BUSINESS",
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": "LEGAL_ENTITY",
        "NUM_OF_EMPLOYEER": str(employee_count),
        "RM_NAME": random.choice(RELATIONSHIP_MANAGERS),
        "FLG_IS_AFFLUENT": is_affluent,
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=12)
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=11)
        ),
        "EMPL_PROPERTY_TYPE": "LEGAL_ENTITY",
        "SHORT_NAME": company_name.replace(" ", "")[:20].upper(),
        "DOCUMENT_NAME": "TAX_CERTIFICATE",
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": company_name.upper(),
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(city),
        "WORKPLACE": company_name,
        "TAX_TYPE": "LEGAL_ENTITY",
        "CUST_MIS_1": "CORP",
        "SHORT_NAME2": company_name[:20].upper(),
        "CHARGE_GROUP": "KORP",
        "CUSTOMER_ADDRESS": make_address(city),
        "CORP_REGISTER_ADDRESS": make_address(city),
        "CORP_REGISTER_COUNTRY": "AZ",
        "CUST_SUB_SEGMENT": (
            "LARGE_CORP"
            if employee_count >= 250
            else "SME"
        ),
        "INCORP_DATE": format_date(incorporation_date),
        "DT_QHT_GROUP": f"GRP_{random.randint(1, 300):04d}",
        "CUST_CLASSIFICATION": (
            "PREMIUM"
            if is_affluent == "Y"
            else "STANDARD"
        ),
        "LOAN_RESP_ID": f"LR{random.randint(1000, 9999)}",
        "FLG_IS_RELATED_PERSON": (
            "Y"
            if random.random() < 0.03
            else "N"
        ),
        "RM_ID_CORP": f"RM{random.randint(1001, 1010)}",
        "GROUP_ID_CORP": f"GRP{random.randint(1, 300):04d}",
    })

    return row


def generate_bank(
    row: dict,
    first_contract_date: date,
    existing_tax_ids: set,
    bank_sequence: int
) -> dict:
    bank_name = (
        f"{random.choice(['Caspian', 'Azer', 'Shirvan', 'Atlas'])} "
        f"Bank ASC {bank_sequence:03d}"
    )

    incorporation_date = random_date(
        date(1992, 1, 1),
        first_contract_date - timedelta(days=365)
    )

    creation_date = random_date(
        incorporation_date,
        first_contract_date
    )

    city = "BAKI"

    row.update({
        "TAX_ID": generate_unique_tax_id(existing_tax_ids),
        "CUSTOMER_TYPE": "B",
        "CUSTOMER_SUB_TYPE": "BANK",
        "FULL_NAME": bank_name,
        "OPEN_REASON": "CORRESPONDENT",
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": "BANK",
        "NUM_OF_EMPLOYEER": str(
            random.randint(200, 2500)
        ),
        "RM_NAME": random.choice(RELATIONSHIP_MANAGERS),
        "FLG_IS_AFFLUENT": "Y",
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=12)
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=11)
        ),
        "EMPL_PROPERTY_TYPE": "BANK",
        "SHORT_NAME": bank_name.replace(" ", "")[:20].upper(),
        "DOCUMENT_NAME": "BANK_LICENSE",
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": bank_name.upper(),
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(city),
        "WORKPLACE": bank_name,
        "TAX_TYPE": "FINANCIAL_INSTITUTION",
        "CUST_MIS_1": "BANK",
        "SHORT_NAME2": bank_name[:20].upper(),
        "SWIFT_CD": random.choice(SWIFT_CODES),
        "CHARGE_GROUP": "BANK",
        "CUSTOMER_ADDRESS": make_address(city),
        "CORP_REGISTER_ADDRESS": make_address(city),
        "CORP_REGISTER_COUNTRY": "AZ",
        "CUST_SUB_SEGMENT": "BANK",
        "INCORP_DATE": format_date(incorporation_date),
        "CUST_CLASSIFICATION": "PREMIUM",
        "LOAN_RESP_ID": f"LR{random.randint(1000, 9999)}",
        "RM_ID_CORP": f"RM{random.randint(1001, 1010)}",
        "GROUP_ID_CORP": f"BNK{bank_sequence:03d}",
    })

    return row


# ============================================================
# VALIDATION
# ============================================================

def validate_customer_rows(
    customer_rows: list,
    contract_customer_ids: set
) -> None:
    customer_numbers = [
        row["CUSTOMER_NO"]
        for row in customer_rows
    ]

    individual_pins = [
        row["PIN"]
        for row in customer_rows
        if row["CUSTOMER_TYPE"] == "I"
    ]

    if len(customer_rows) != NUMBER_OF_CUSTOMERS:
        raise ValueError("Customer row count is incorrect.")

    if len(customer_numbers) != len(set(customer_numbers)):
        raise ValueError("Duplicate CUSTOMER_NO / CIF values exist.")

    if any(not customer_no for customer_no in customer_numbers):
        raise ValueError("Blank CUSTOMER_NO / CIF values exist.")

    if len(individual_pins) != len(set(individual_pins)):
        raise ValueError("Duplicate individual PIN values exist.")

    if any(not pin for pin in individual_pins):
        raise ValueError("An individual customer has no PIN.")

    if any(
        row["PIN"]
        for row in customer_rows
        if row["CUSTOMER_TYPE"] in {"C", "B"}
    ):
        raise ValueError(
            "A company or bank incorrectly has a PIN."
        )

    missing_customers = (
        contract_customer_ids - set(customer_numbers)
    )

    if missing_customers:
        raise ValueError(
            "Some contract CUSTOMER_ID values have no "
            "matching CUSTOMER_NO."
        )


def write_csv(file_path: Path, rows: list) -> None:
    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=CUSTOMER_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main():
    (
        contract_customer_ids,
        first_contract_date_by_customer,
        product_codes_by_customer
    ) = load_contract_customer_details()

    customer_numbers = build_customer_numbers(
        contract_customer_ids
    )

    existing_pins = set()
    existing_documents = set()
    existing_tax_ids = set()

    customer_rows = []

    individual_count = 0
    corporate_count = 0
    bank_count = 0

    for row_number, customer_no in enumerate(
        customer_numbers,
        start=1
    ):
        first_contract_date = first_contract_date_by_customer.get(
            customer_no,
            date(2026, 5, 31)
        )

        customer_products = product_codes_by_customer.get(
            customer_no,
            set()
        )

        customer_type = choose_customer_type(customer_products)

        row = make_base_row(
            customer_no=customer_no,
            generated_id=100_000_000 + row_number
        )

        if customer_type == "I":
            individual_count += 1

            row = generate_individual(
                row=row,
                first_contract_date=first_contract_date,
                existing_pins=existing_pins,
                existing_documents=existing_documents
            )

        elif customer_type == "C":
            corporate_count += 1

            row = generate_company(
                row=row,
                first_contract_date=first_contract_date,
                existing_tax_ids=existing_tax_ids,
                company_sequence=corporate_count
            )

        else:
            bank_count += 1

            row = generate_bank(
                row=row,
                first_contract_date=first_contract_date,
                existing_tax_ids=existing_tax_ids,
                bank_sequence=bank_count
            )

        customer_rows.append(row)

    validate_customer_rows(
        customer_rows=customer_rows,
        contract_customer_ids=contract_customer_ids
    )

    full_output_file = (
        RAW_DATA_FOLDER / "rs_customers_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER / "rs_customers_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER / "rs_customers_generation_summary.json"
    )

    write_csv(full_output_file, customer_rows)
    write_csv(sample_output_file, customer_rows[:100])

    summary = {
        "table_name": "RS_CUSTOMERS",
        "generated_rows": len(customer_rows),
        "contract_customer_ids_found": len(contract_customer_ids),
        "individual_customers": individual_count,
        "corporate_customers": corporate_count,
        "bank_customers": bank_count,
        "unique_customer_numbers": len(
            {row["CUSTOMER_NO"] for row in customer_rows}
        ),
        "unique_individual_pins": len(existing_pins),
        "contract_customer_ids_without_match": 0,
        "join_rule": (
            "RS_CONTRACTS.CUSTOMER_ID "
            "= RS_CUSTOMERS.CUSTOMER_NO"
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("Customer generation completed successfully.")
    print(f"Generated customers: {len(customer_rows):,}")
    print(f"Individuals: {individual_count:,}")
    print(f"Corporate customers: {corporate_count:,}")
    print(f"Banks: {bank_count:,}")
    print(
        "Contract-to-customer unmatched IDs: 0"
    )
    print(f"Full dataset: {full_output_file}")
    print(f"Sample dataset: {sample_output_file}")


if __name__ == "__main__":
    main()

Customer generation completed successfully.
Generated customers: 15,000
Individuals: 11,363
Corporate customers: 3,482
Banks: 155
Contract-to-customer unmatched IDs: 0
Full dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_customers_base.csv
Sample dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_customers_base_sample.csv
